# 🏗️ Notebook 1: Typeahead / Autocomplete — Requirements & Architecture

## 🛠️ Setup

```bash
cd 06-system-designs/typeahead-autocomplete
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## What we're designing

As the user types `"goo"` → suggest `"google", "good morning", "google docs"`.
Under **20ms per keystroke**, for millions of users.

### Functional requirements
- Given a prefix, return top-K suggestions **ordered by popularity** (weighted).
- Suggestions update as new queries trend.
- Personalization (nice-to-have, not here).

### Non-functional
- **Very low latency**: <20ms at p99.
- **High QPS**: every keystroke is a request; power-users type 5 chars/sec.
- Must gracefully handle typos and emoji.

### Core data structure: **trie** (prefix tree)

```
       (root)
       /  |  \
      g   a   b
     /    |    \
    o     p     e
   /|     |     \
  o o     p     e
  |  |    |     |
  d  gle  le    r
```

Each node holds a "top-K for this prefix" cache. Lookup is just tree walk + read cache.


## High-level architecture

```
     [client]
        │ each keystroke
        ▼
   ┌─────────┐
   │ Edge /  │◀─── cache popular prefixes
   │ CDN     │
   └────┬────┘
        │ miss
        ▼
   ┌─────────┐        ┌───────────────┐
   │ Suggest │──────▶│ Trie service  │   (in-memory, sharded by prefix)
   │ Service │        └───────┬───────┘
   └────┬────┘                │
        │                     ▼
        │              ┌────────────┐
        │              │ Query logs │ (kafka stream)
        │              └─────┬──────┘
        │                    │ every N mins
        │                    ▼
        │             ┌─────────────┐
        │             │ Aggregator  │ (Spark/Flink)
        │             │ → new trie  │
        │             └─────────────┘
        ▼
   Return top-K
```

**Key idea**: the trie is rebuilt periodically from query logs (every 5–15 min).
Live edits to the trie are hard to do safely; batched snapshot swaps are simpler.
